In [41]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.activations import linear, relu, sigmoid

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import math
from collections import Counter

In [42]:
df = pd.read_csv('cryptography_dataset_processed.csv')

In [43]:
df.head()

,Algorithm,Key,Ciphertext,Key Length (bits),Ciphertext Length (bytes)
0,AES,ef7d8f79011629e953bf4f7738317bb5,5bff23c6f03e37c1,0.036866,0.000000
1,DES,3453d3b1750432cf,b1366a30a0c3b8f68093c601b061180d,0.000000,0.032258
2,Triple DES,df3445340794dcce3d07bcc1ad375ece524ab5d001073b08,c0917f1c9df3d9b3965ac26779314db5,0.073733,0.032258
3,RSA,-----BEGIN PUBLIC KEY-----\nMIIBIjANBgkqhkiG9w...,c20b03112d92b17d2daaac7e868fd6b697f436a03f8dbb...,1.000000,1.000000
4,Blowfish,1db0711ab8922217c4bf74c613073d8c,428aba00a39ee506e87c1b2924e8e219,0.036866,0.032258


In [44]:
df.shape

(120000, 5)

In [45]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 5 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Algorithm                  120000 non-null  str    
 1   Key                        120000 non-null  str    
 2   Ciphertext                 120000 non-null  str    
 3   Key Length (bits)          120000 non-null  float64
 4   Ciphertext Length (bytes)  120000 non-null  float64
dtypes: float64(2), str(3)
memory usage: 4.6 MB


In [46]:
df.describe()

,Key Length (bits),Ciphertext Length (bytes)
count,120000.000000,120000.000000
mean,0.157450,0.099164
std,0.270787,0.271758
min,0.000000,0.000000
25%,0.036866,0.012097
50%,0.036866,0.012097
75%,0.110599,0.032258
max,1.000000,1.000000


In [47]:
df.isnull().sum()  

Algorithm                    0
Key                          0
Ciphertext                   0
Key Length (bits)            0
Ciphertext Length (bytes)    0
dtype: int64

In [48]:
def calculate_entropy(hex_str):
    data = bytes.fromhex(hex_str) if isinstance(hex_str, str) else hex_str
    if not data:
        return 0.0
    counts = Counter(data)
    total = len(data)
    return -sum((count / total) * math.log2(count / total) for count in counts.values())

df['entropy'] = df['Ciphertext'].apply(calculate_entropy)

In [49]:
def calculate_byte_variance(hex_str):
    try:
        data = bytes.fromhex(hex_str) if isinstance(hex_str, str) else hex_str
        if not data:
            return 0.0
        return float(np.var(list(data)))
    except ValueError:
        return 0.0

# Apply byte variance to your dataframe
df['byte_variance'] = df['Ciphertext'].apply(calculate_byte_variance)

In [50]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.utils import to_categorical

# Encode Target
le = LabelEncoder()
y_encoded = le.fit_transform(df['Algorithm'])
y_categorical = to_categorical(y_encoded)

# ADD 'byte_variance' TO YOUR FEATURES LIST HERE:
features = ['Key Length (bits)', 'Ciphertext Length (bytes)', 'entropy', 'byte_variance']
X = df[features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_categorical, test_size=0.2, random_state=42
)

In [51]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, np.argmax(y_train, axis=1))

y_pred_rf = rf_model.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(np.argmax(y_test, axis=1), y_pred_rf))

Random Forest Accuracy: 0.718125


In [52]:
from sklearn.metrics import classification_report
# Flatten y_test from one-hot categorical format to 1D integer labels
y_test_flat = np.argmax(y_test, axis=1)
print(classification_report(y_test_flat, y_pred_rf, target_names=le.classes_))

              precision    recall  f1-score   support

         AES       0.50      0.50      0.50      2014
    Blowfish       1.00      1.00      1.00      1982
    Camellia       0.20      0.20      0.20      1975
    ChaCha20       1.00      1.00      1.00      2066
         DES       1.00      1.00      1.00      1935
         ECC       1.00      1.00      1.00      2023
     ElGamal       0.18      0.15      0.17      2044
        GOST       1.00      1.00      1.00      2066
         RC4       0.50      0.50      0.50      1988
         RSA       1.00      1.00      1.00      1972
     Serpent       0.20      0.24      0.22      1901
  Triple DES       1.00      1.00      1.00      2034

    accuracy                           0.72     24000
   macro avg       0.72      0.72      0.72     24000
weighted avg       0.72      0.72      0.72     24000



In [53]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

num_classes = y_categorical.shape[1]
input_dim = X_train.shape[1]

model = Sequential([
    Dense(64, activation='relu', input_shape=(input_dim,)),
    BatchNormalization(),
    Dropout(0.3),
    
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

C:\Users\polin\anaconda3\envs\tfenv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [54]:
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# 1. Convert one-hot encoded y_train to 1D integer labels first
y_train_integers = np.argmax(y_train, axis=1)

# 2. Compute class weights using TRAINING data (y_train_integers)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_integers),
    y=y_train_integers
)
# class_weight_dict = dict(enumerate(class_weights))
class_weight_dict = {class_id: weight ** 0.5 for class_id, weight in enumerate(class_weights)}

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# 3. Train with class weights
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[early_stop]
)

# 4. Evaluate on test set (y_test_classes is defined here)
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

print(classification_report(y_test_classes, y_pred_classes, target_names=le.classes_))

Epoch 1/100
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.7113 - loss: 0.5754 - val_accuracy: 0.7513 - val_loss: 0.3941
Epoch 2/100
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.7459 - loss: 0.4098 - val_accuracy: 0.7513 - val_loss: 0.3921
Epoch 3/100
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.7483 - loss: 0.4021 - val_accuracy: 0.7491 - val_loss: 0.3904
Epoch 4/100
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.7480 - loss: 0.4004 - val_accuracy: 0.7533 - val_loss: 0.3883
Epoch 5/100
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.7480 - loss: 0.4011 - val_accuracy: 0.7515 - val_loss: 0.3916
Epoch 6/100
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.7463 - loss: 0.4012 - val_accuracy: 0.7516 - val_loss: 0.3882
Epoch 7/100
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.7462 - loss: 0.3995 - val_accuracy: 0.7541 - val_loss: 0.3883
Epoch 8/100
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.7465 - 

C:\Users\polin\anaconda3\envs\tfenv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\polin\anaconda3\envs\tfenv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\polin\anaconda3\envs\tfenv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape